# Standalone CLCM Replication (Colab)

This notebook is standalone: it does not import local `src/*.py` files.

Only requirement: FER2013 data must exist in Colab (local upload, Drive, or Kaggle download).

In [ ]:
!pip -q install torch torchvision tqdm scikit-learn kagglehub

import os
import random
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import kagglehub

SEED = 42
EPOCHS = 100
BATCH_SIZE = 64
LR = 1e-3
WEIGHT_DECAY = 1e-4
VAL_FRACTION = 0.10
SAVE_DIR = Path('/content/saved_models')
SAVE_DIR.mkdir(parents=True, exist_ok=True)
BEST_MODEL_PATH = SAVE_DIR / 'clcm_best_weights.pth'

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

In [ ]:
EMOTIONS = ['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise', 'neutral']
IMAGE_EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.webp'}

# Option A: if dataset already uploaded/mounted, set this path directly.
# Example: DATA_ROOT = Path('/content/fer2013')
DATA_ROOT = Path('/content/fer2013')

if not DATA_ROOT.exists():
    print('DATA_ROOT not found, downloading FER2013 from KaggleHub...')
    downloaded_path = Path(kagglehub.dataset_download('msambare/fer2013'))
    print('Downloaded to:', downloaded_path)

    candidates = [downloaded_path, *downloaded_path.rglob('fer2013')]
    resolved = None
    for c in candidates:
        if (c / 'train').is_dir() and (c / 'test').is_dir():
            resolved = c
            break
    if resolved is None:
        raise RuntimeError('Could not resolve FER2013 train/test folders. Set DATA_ROOT manually.')
    DATA_ROOT = resolved

TRAIN_ROOT = DATA_ROOT / 'train'
TEST_ROOT = DATA_ROOT / 'test'

if not TRAIN_ROOT.is_dir() or not TEST_ROOT.is_dir():
    raise RuntimeError(f'Expected train/test folders under {DATA_ROOT}')

print('Resolved DATA_ROOT =', DATA_ROOT)

In [ ]:
def list_class_images(root: Path):
    out = {}
    for cls in EMOTIONS:
        cls_dir = root / cls
        files = []
        if cls_dir.is_dir():
            for p in sorted(cls_dir.iterdir()):
                if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
                    files.append(p)
        out[cls] = files
    return out

class_image_map = list_class_images(TRAIN_ROOT)
train_files, train_labels = [], []
val_files, val_labels = [], []

for idx, cls in enumerate(EMOTIONS):
    files = class_image_map[cls]
    rng = np.random.default_rng(SEED + idx)
    perm = rng.permutation(len(files))
    files = [files[i] for i in perm]

    val_count = int(round(len(files) * VAL_FRACTION))
    val_count = max(1, min(val_count, max(len(files) - 1, 1))) if len(files) > 1 else 0

    val_split = files[:val_count]
    train_split = files[val_count:]

    train_files.extend(train_split)
    train_labels.extend([idx] * len(train_split))
    val_files.extend(val_split)
    val_labels.extend([idx] * len(val_split))

test_files, test_labels = [], []
for idx, cls in enumerate(EMOTIONS):
    cls_dir = TEST_ROOT / cls
    if cls_dir.is_dir():
        for p in sorted(cls_dir.iterdir()):
            if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
                test_files.append(p)
                test_labels.append(idx)

print(f'Train samples: {len(train_files)}')
print(f'Val samples:   {len(val_files)}')
print(f'Test samples:  {len(test_files)}')

class FERImageDataset(Dataset):
    def __init__(self, files, labels, transform=None):
        self.files = files
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert('L')
        if self.transform is not None:
            img = self.transform(img)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return img, label

train_tfms = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

eval_tfms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

train_ds = FERImageDataset(train_files, train_labels, transform=train_tfms)
val_ds = FERImageDataset(val_files, val_labels, transform=eval_tfms)
test_ds = FERImageDataset(test_files, test_labels, transform=eval_tfms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.depthwise = nn.Conv2d(in_channels, in_channels, 3, stride=stride, padding=1, groups=in_channels, bias=False)
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.pointwise = nn.Conv2d(in_channels, out_channels, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU6(inplace=True)

    def forward(self, x):
        x = self.relu(self.bn1(self.depthwise(x)))
        x = self.relu(self.bn2(self.pointwise(x)))
        return x

class CLCM(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        self.init_conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU6(inplace=True),
        )
        self.features = nn.Sequential(
            DepthwiseSeparableConv(32, 64, stride=2),
            DepthwiseSeparableConv(64, 128, stride=2),
            DepthwiseSeparableConv(128, 256, stride=2),
            DepthwiseSeparableConv(256, 256, stride=1),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(nn.Dropout(0.5), nn.Linear(256, num_classes))

    def forward(self, x):
        x = self.init_conv(x)
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

model = CLCM(num_classes=7).to(device)
print(f'Trainable params: {count_parameters(model):,}')
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

In [ ]:
def run_eval(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            total_loss += loss.item() * x.size(0)
            preds = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(y.cpu().numpy().tolist())
    avg_loss = total_loss / max(len(loader.dataset), 1)
    acc = 100.0 * accuracy_score(all_labels, all_preds) if all_labels else 0.0
    return avg_loss, acc, all_labels, all_preds

best_val_acc = 0.0
history = []

for epoch in range(EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        total += y.size(0)
        correct += (preds == y).sum().item()

    train_loss = running_loss / max(total, 1)
    train_acc = 100.0 * correct / max(total, 1)

    val_loss, val_acc, _, _ = run_eval(model, val_loader)
    scheduler.step(val_loss)

    history.append((epoch + 1, train_loss, train_acc, val_loss, val_acc))
    print(f'Epoch [{epoch+1:03d}/{EPOCHS}] | Train Loss: {train_loss:.4f} - Train Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f} - Val Acc: {val_acc:.2f}%')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f'  New best val acc! saved -> {BEST_MODEL_PATH}')

print(f'Best validation accuracy: {best_val_acc:.2f}%')

In [ ]:
best_model = CLCM(num_classes=7).to(device)
best_model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
test_loss, test_acc, y_true, y_pred = run_eval(best_model, test_loader)

print('\n' + '='*50)
print(f'TEST ACCURACY: {test_acc:.2f}%')
print('='*50)

cm = confusion_matrix(y_true, y_pred)
print('Confusion Matrix:\n', cm)
print('\nClassification Report:\n')
print(classification_report(y_true, y_pred, target_names=EMOTIONS, digits=4))

## Runtime Expectation (A100)

- 50 epochs: usually around 8 to 18 minutes.
- 100 epochs: usually around 16 to 35 minutes.

Actual time depends mostly on data loading speed and whether the dataset is cached in the Colab runtime.